# MallWorld Data Exploration: Looking for Non-Obvious Patterns

**Goal**: Explore the data to find patterns that could discriminate between competing hypotheses—WITHOUT pre-deciding what to look for.

The key question: What's in this data that might surprise us? What relationships exist that common sense would NOT predict?

Great question. Let me explain what these massive χ² values actually mean and what that implies for MallWorld research.

## What the Massive Chi-Square Values Mean

### The Basic Logic

Chi-square tests whether two variables are **independent** (unrelated) or **associated** (systematically co-occur). The null hypothesis is: "These things are randomly paired—no relationship."

| Finding | χ² | p-value | What It Means |
|---------|-----|---------|---------------|
| Location-atmosphere | 1,677.05 | < 10⁻¹²⁶ | The odds this pattern is random chance: **1 in 10¹²⁶** |
| Entity-atmosphere | 792.02 | < 10⁻¹⁰⁸ | The odds this pattern is random chance: **1 in 10¹⁰⁸** |
| Vertical-atmosphere | 143.1 | < 10⁻²¹ | The odds this pattern is random chance: **1 in 10²¹** |

For comparison:
- There are approximately 10⁸⁰ atoms in the observable universe
- 10⁻¹²⁶ is **46 orders of magnitude smaller** than picking one specific atom at random

### What This Actually Tells Us

**The patterns are real, not noise.** When you see χ² = 1,677 with p < 10⁻¹²⁶, you're looking at structure that cannot plausibly arise from random dream generation. Something is organizing this data.

But here's the key insight: **chi-square tells you the pattern exists, not what causes it.**

The massive values establish:
1. ✅ MallWorld dreams have **non-random internal structure**
2. ✅ Location types systematically predict atmospheres
3. ✅ Entity types systematically predict emotional environments
4. ✅ Vertical position systematically predicts atmosphere

They do NOT establish:
- ❓ WHY these patterns exist
- ❓ Whether the patterns are cultural (learned) or ontological (inherent)
- ❓ Whether dreamers are perceiving something real or constructing something consistent

### The Research Opportunity

This is where MallWorld gets interesting. The chi-square values have done their job—they've proven structure exists. Now the question becomes: **What kind of structure is it?**

There are three competing explanations:

| Explanation | Prediction | How to Test |
|-------------|------------|-------------|
| **Cultural Construction** | Patterns should vary by demographic (age, culture, religion) | Cross-cultural analysis; demographic stratification |
| **Psychological Projection** | Patterns should correlate with individual traits (anxiety, trauma) | Within-dreamer consistency; psychological profiling |
| **Correspondential Ontology** | Patterns should be universal regardless of demographics; same structure as NDE/myth | Cross-cultural consistency; comparison to independent consciousness phenomena |

### Where to Go Deeper

Given that structure exists, the next analytical steps would be:

1. **Cross-Cultural Consistency Test**: Do non-Western MallWorld dreamers show the same location-atmosphere mappings? If correspondence is ontological, it shouldn't vary by culture.

2. **Within-Dreamer Longitudinal Analysis**: Do individual dreamers maintain consistent correspondential patterns across multiple dreams? This tests whether the structure is perceived (consistent) vs constructed (variable).

3. **Comparison to Independent Phenomena**: How closely do MallWorld entity patterns match NDE entity patterns? Identical function differentiation across dream and near-death states would be strong evidence for perception of the same reality.

4. **Prediction Testing**: The framework predicts specific correspondences (fire = love/passion, water = truth, etc.). Can we find explicit symbolic content in dreams that tests these predictions?

5. **Effect Size Analysis**: Chi-square tells us patterns exist but not how STRONG they are. Cramér's V or similar measures would tell us what percentage of atmosphere variation is explained by location/entity type.

Would you like me to explore any of these directions, or look at what data you already have that could support these deeper analyses?

In [4]:
import json
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu
from collections import Counter, defaultdict
from pathlib import Path
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD RAW STRUCTURED DATA (Full control over all fields)
# ============================================================================

def load_mallworld_data(structured_path: Path) -> List[Dict]:
    """Load all extracted Mall World JSON files."""
    records = []
    for json_file in sorted(structured_path.glob('*.json')):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                record = json.load(f)
                records.append(record)
        except Exception as e:
            print(f"Error loading {json_file.name}: {e}")
    return records

# Load data
structured_path = Path('../structured')
raw_records = load_mallworld_data(structured_path)
print(f"✓ Loaded {len(raw_records):,} total records")

# Filter to actual dream reports
VALID_POST_TYPES = {'dream_report', 'dream_report_with_map'}

def is_valid_dream_report(record: Dict) -> bool:
    ext = record.get('extraction', {}) or {}
    classification = ext.get('classification', {}) or {}
    post_type = classification.get('post_type', '')
    if post_type not in VALID_POST_TYPES:
        return False
    if classification.get('should_skip_extraction', False):
        return False
    locations = ext.get('locations', [])
    if not locations or len(locations) == 0:
        return False
    return True

valid_records = [r for r in raw_records if is_valid_dream_report(r)]
print(f"✓ After filtering: {len(valid_records):,} valid dream reports")

✓ Loaded 3,732 total records
✓ After filtering: 1,926 valid dream reports


In [5]:
# ============================================================================
# FLATTEN DATA INTO DATAFRAMES (with full field access)
# ============================================================================

def get_dream_id(record):
    """Extract dream ID from record."""
    return record.get('source_url', '').split('/')[-2] if record.get('source_url') else record.get('source_file', '')

# Build locations DataFrame with ALL fields
locations_data = []
for record in valid_records:
    dream_id = get_dream_id(record)
    ext = record.get('extraction', {}) or {}
    meta = ext.get('meta', {}) or {}
    author = meta.get('author', {}) or {}
    
    for loc in ext.get('locations', []):
        qualities = loc.get('qualities', {}) or {}
        position = loc.get('position', {}) or {}
        
        locations_data.append({
            'dream_id': dream_id,
            'location_id': loc.get('location_id'),
            'location_type': loc.get('location_type', 'unspecified'),
            'location_name': loc.get('location_name'),
            # Position
            'vertical_position': position.get('vertical', 'not_mentioned'),
            'horizontal_position': position.get('horizontal', 'not_mentioned'),
            'cardinal_direction': position.get('cardinal', 'not_mentioned'),
            # Core qualities
            'light': qualities.get('light', 'not_mentioned'),
            'light_temperature': qualities.get('light_temperature', 'not_mentioned'),
            'time_of_day': qualities.get('time_of_day', 'not_mentioned'),
            'state': qualities.get('state', 'not_mentioned'),
            'atmosphere': qualities.get('atmosphere', 'not_mentioned'),
            'reality_stability': qualities.get('reality_stability', 'not_mentioned'),
            'crowding': qualities.get('crowding', 'not_mentioned'),
            'crowd_behavior': qualities.get('crowd_behavior', 'not_mentioned'),
            'cleanliness': qualities.get('cleanliness', 'not_mentioned'),
            'privacy_status': qualities.get('privacy_status', 'not_mentioned'),
            # Dreamer response
            'affective_response': loc.get('affective_response', 'not_mentioned'),
            'somatic_response': loc.get('somatic_response', 'none'),
            'dreamer_role': loc.get('dreamer_role', 'not_mentioned'),
            'visit_order': loc.get('visit_order'),
            # Demographics from author (THIS IS THE KEY PART)
            'author_gender': author.get('gender', 'not_mentioned'),
            'author_age_range': author.get('age_range', 'not_mentioned'),
            # Dream-level meta
            'recurrence': meta.get('recurrence', 'not_mentioned'),
            'dreamer_identity': meta.get('dreamer_identity', 'not_mentioned'),
        })

df_locations = pd.DataFrame(locations_data)

# Build entities DataFrame
entities_data = []
for record in valid_records:
    dream_id = get_dream_id(record)
    ext = record.get('extraction', {}) or {}
    meta = ext.get('meta', {}) or {}
    author = meta.get('author', {}) or {}
    
    for interaction in ext.get('interactions', []):
        loc_id = interaction.get('location_id')
        for entity in interaction.get('entities_involved', []):
            entities_data.append({
                'dream_id': dream_id,
                'location_id': loc_id,
                'entity_type': entity.get('entity_type', 'other'),
                'demeanor': entity.get('demeanor', 'not_mentioned'),
                'role': entity.get('role', 'not_mentioned'),
                'authority_nature': entity.get('authority_nature', 'not_mentioned'),
                'is_known': entity.get('is_known', 'not_mentioned'),
                'is_recurring': entity.get('is_recurring', 'not_mentioned'),
                # Demographics
                'author_gender': author.get('gender', 'not_mentioned'),
                'author_age_range': author.get('age_range', 'not_mentioned'),
            })

df_entities = pd.DataFrame(entities_data)

# Build connections DataFrame
connections_data = []
for record in valid_records:
    dream_id = get_dream_id(record)
    ext = record.get('extraction', {}) or {}
    
    for conn in ext.get('connections', []):
        connections_data.append({
            'dream_id': dream_id,
            'from_location_id': conn.get('from_location_id'),
            'to_location_id': conn.get('to_location_id'),
            'connection_type': conn.get('connection_type', 'unspecified'),
            'transit_mode': conn.get('transit_mode', 'not_mentioned'),
            'direction': conn.get('direction', 'unknown'),
            'difficulty': conn.get('difficulty', 'not_mentioned'),
            'outcome': conn.get('outcome', 'not_mentioned'),
        })

df_connections = pd.DataFrame(connections_data)

# Build interactions DataFrame
interactions_data = []
for record in valid_records:
    dream_id = get_dream_id(record)
    ext = record.get('extraction', {}) or {}
    
    for interaction in ext.get('interactions', []):
        interactions_data.append({
            'dream_id': dream_id,
            'location_id': interaction.get('location_id'),
            'interaction_type': interaction.get('interaction_type', 'other'),
            'outcome': interaction.get('outcome', 'not_mentioned'),
            'failure_type': interaction.get('failure_type', 'not_applicable'),
            'n_entities': len(interaction.get('entities_involved', [])),
        })

df_interactions = pd.DataFrame(interactions_data)

print(f"\n✓ DataFrames built:")
print(f"  Locations:    {len(df_locations):,}")
print(f"  Entities:     {len(df_entities):,}")
print(f"  Connections:  {len(df_connections):,}")
print(f"  Interactions: {len(df_interactions):,}")


✓ DataFrames built:
  Locations:    8,707
  Entities:     3,704
  Connections:  4,600
  Interactions: 6,075
